# Проверка: пересечение H2O и CRM по ФП

**Цель:** выяснить, какая доля записей H2O (с `ref_book_fp_id`) присутствует в CRM,
и нужны ли нам записи H2O, которых нет в CRM.

**Связка:** `h2o.crm_fp_id` ↔ `crm.ROW_ID`  
**Фильтр CRM:** `VAL = 'H2.0'` (только записи из источника H2O)

## 1. Загрузка данных

In [ ]:
import pandas as pd
import numpy as np

# --- H2O ---
df_h2o = pd.read_excel("../sources/data_h20.xlsx", dtype=str)
df_h2o.columns = df_h2o.columns.str.strip()
print(f"H2O: {len(df_h2o):,} строк")

# --- CRM ---
df_crm = pd.read_csv("../sources/data_crm.csv", sep=";", encoding="utf-8-sig", dtype=str, low_memory=False)
df_crm.columns = df_crm.columns.str.strip()
print(f"CRM: {len(df_crm):,} строк")

## 2. Уникальные значения VAL в CRM

Смотрим, какие источники есть в CRM и как именно называется H2O.

In [ ]:
print("Уникальные значения VAL (источник) в CRM:")
print(df_crm["VAL"].value_counts(dropna=False).to_string())

## 3. Фильтрация CRM: только записи из H2O

⚠️ **Проверьте значение фильтра** после просмотра вывода выше. Если источник называется иначе (например `'H2O'`, `'h2.0'`, `'Н2.0'` с кириллической Н), скорректируйте.

In [ ]:
# Подставьте точное значение из вывода выше
H2O_SOURCE_NAME = "H2.0"

crm_h2o = df_crm[df_crm["VAL"].str.strip() == H2O_SOURCE_NAME].copy()
print(f"CRM записей с VAL='{H2O_SOURCE_NAME}': {len(crm_h2o):,}")
print(f"CRM записей из других источников:       {len(df_crm) - len(crm_h2o):,}")

## 4. Пересечение по ключу: h2o.crm_fp_id ↔ crm.ROW_ID

In [ ]:
# Чистим ключи связки
h2o_keys = (
    df_h2o["crm_fp_id"]
    .dropna()
    .astype(str).str.strip()
)
h2o_keys = set(h2o_keys[h2o_keys != ""])

crm_keys = (
    crm_h2o["ROW_ID"]
    .dropna()
    .astype(str).str.strip()
)
crm_keys = set(crm_keys[crm_keys != ""])

# Пересечения
both = h2o_keys & crm_keys
only_h2o = h2o_keys - crm_keys
only_crm = crm_keys - h2o_keys

print(f"Уникальных crm_fp_id в H2O:    {len(h2o_keys):,}")
print(f"Уникальных ROW_ID в CRM (H2O):  {len(crm_keys):,}")
print(f"")
print(f"Совпадают (есть в обоих):        {len(both):,}")
print(f"Только в H2O (нет в CRM):        {len(only_h2o):,}")
print(f"Только в CRM (нет в H2O):        {len(only_crm):,}")

## 5. Анализ записей H2O, которых нет в CRM

Кто эти «лишние» записи? Есть ли у них `ref_book_fp_id`?

In [ ]:
# Записи H2O, чей crm_fp_id НЕ нашёлся в CRM
h2o_not_in_crm = df_h2o[
    df_h2o["crm_fp_id"].astype(str).str.strip().isin(only_h2o)
].copy()

# Записи H2O вообще без crm_fp_id (пустые)
h2o_no_key = df_h2o[
    df_h2o["crm_fp_id"].isna() |
    (df_h2o["crm_fp_id"].astype(str).str.strip() == "")
]

# Записи H2O, которые нашлись в CRM
h2o_in_crm = df_h2o[
    df_h2o["crm_fp_id"].astype(str).str.strip().isin(both)
]

print(f"H2O записей: всего {len(df_h2o):,}")
print(f"  — нашлись в CRM:           {len(h2o_in_crm):,}")
print(f"  — crm_fp_id есть, но нет в CRM: {len(h2o_not_in_crm):,}")
print(f"  — crm_fp_id пустой:        {len(h2o_no_key):,}")
print(f"  — итого «не в CRM»:        {len(h2o_not_in_crm) + len(h2o_no_key):,}")

In [ ]:
# Есть ли ref_book_fp_id у «лишних» записей?
print("ref_book_fp_id у записей H2O, которых нет в CRM:")
not_in_crm_all = pd.concat([h2o_not_in_crm, h2o_no_key])
n_with_fp = not_in_crm_all["ref_book_fp_id"].notna().sum()
n_without_fp = not_in_crm_all["ref_book_fp_id"].isna().sum()
print(f"  С ref_book_fp_id:    {n_with_fp:,}")
print(f"  Без ref_book_fp_id:  {n_without_fp:,}")

if n_with_fp > 0:
    print(f"\nТипы ФП (ref_book_fp_id) у 'лишних':")
    print(not_in_crm_all["ref_book_fp_id"].value_counts().head(20).to_string())

## 6. Анализ пропусков ref_book_fp_id: связь с наличием в CRM

In [ ]:
# Проверяем гипотезу: пропуски ref_book_fp_id = записи, которых нет в CRM?
df_h2o["_in_crm"] = df_h2o["crm_fp_id"].astype(str).str.strip().isin(both)
df_h2o["_has_fp_type"] = df_h2o["ref_book_fp_id"].notna()

cross = pd.crosstab(
    df_h2o["_in_crm"].map({True: "Есть в CRM", False: "Нет в CRM"}),
    df_h2o["_has_fp_type"].map({True: "ref_book_fp_id есть", False: "ref_book_fp_id пусто"}),
    margins=True,
)
print("Кросс-таблица: наличие в CRM × наличие ref_book_fp_id")
display(cross)

# Очищаем вспомогательные колонки
df_h2o.drop(columns=["_in_crm", "_has_fp_type"], inplace=True)

## 7. Вывод

Заполните после просмотра результатов:

- Записей H2O, которых нет в CRM: **___**
- Из них с ref_book_fp_id: **___**
- **Решение:** если «лишние» записи H2O без ref_book_fp_id и/или без crm_fp_id — они не нужны для анализа, можно безопасно отбросить.